# 04. 변수 추가, 모델 변경, Kaggle 제출 결과 비교

2026-07-22 기록. Sex·Pclass·FamilySize·AgeGroup을 기준으로
Fare·Embarked 추가, Gradient Boosting, threshold를 차례로 비교했다.
높은 점수 자체보다 검증 결과와 제출 결과가 다를 때 무엇을 결론 내릴 수 있는지 살폈다.

각 단계 안에서는 나머지 조건을 고정했다. AgeGroup 결측은 03과 같이 학습 fold의 최빈값으로 처리한다.


> **기록 상태**: 2026년 7월 개인 실험을 2026년 10월 공개용으로 정리한 노트북입니다.
> 2026-10-06 정리본을 Kaggle 원자료(`train.csv`, `test.csv`)로 Google Colab에서 처음부터 다시 실행했고,
> 정리 전 원본에 저장돼 있던 표와 수치가 모두 같은 값으로 재현됨을 확인했습니다. 실행 환경은 마지막 셀에 기록했습니다.
> 원본·수정 범위는 [기록 정리 내역](../reports/provenance.md)에 있습니다.


## 1. 라이브러리와 데이터 불러오기


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Ready")


Ready


In [ ]:
from pathlib import Path
import os

# 로컬: 저장소 루트 또는 notebooks/에서 실행. Kaggle: Titanic 데이터를 연결.
# Colab: train.csv와 test.csv를 /content에 올린 뒤 실행.
# 다른 위치를 쓰려면 TITANIC_DATA_DIR에 train.csv/test.csv가 있는 폴더를 지정한다.
configured_dir = os.environ.get("TITANIC_DATA_DIR")
if configured_dir:
    candidates = [Path(configured_dir)]
else:
    candidates = [
        Path("data"), Path("../data"),
        Path("/kaggle/input/competitions/titanic"),
        Path("/kaggle/input/titanic"),
        Path("/content"),
    ]
data_dir = next(
    (p for p in candidates if (p / "train.csv").is_file() and (p / "test.csv").is_file()),
    None,
)
if data_dir is None:
    raise FileNotFoundError(
        "Titanic train.csv와 test.csv를 data/에 넣거나 TITANIC_DATA_DIR을 지정하세요."
    )
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
assert "Survived" in train.columns and "Survived" not in test.columns
assert train["PassengerId"].is_unique and test["PassengerId"].is_unique
print("train:", train.shape, "test:", test.shape)


## 2. Feature 추가 가치 비교

아직 사용하지 않은 변수는 `Name`, `Ticket`, `Fare`, `Embarked`였다.

- `Name`과 `Ticket` 원문은 고유값이 많아 이번 실험에서는 제외한다.
- `Fare`는 `Pclass`와 정보가 겹칠 수 있다는 가설을 확인한다.
- `Embarked`가 기존 feature에 추가 정보를 제공하는지 확인한다.
- `Name`에서 `Title`을 추출하는 실험은 이후 후보로 남겨 둔다.


In [3]:
# Fare와 Embarked의 기본 분포 확인
print(train["Fare"].describe())
print(train["Embarked"].describe())

train["Embarked"].value_counts()


count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64
count     889
unique      3
top         S
freq      644
Name: Embarked, dtype: object


,count
Embarked,
S,644
C,168
Q,77


In [4]:
# Baseline: Sex, Pclass, FamilySize, AgeGroup
def create_features(data):
    result = data.copy()

    # 본인을 포함한 전체 가족 규모
    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1

    # Age 원값 대신 사용할 연령 구간
    result["AgeGroup"] = pd.cut(
        result["Age"],
        bins=[0, 6, 20, 40, 60, np.inf],
        labels=["Preschool", "Teen", "20-39", "40-59", "60+"],
        right=False,
    )
    return result


train_fe = create_features(train)
test_fe = create_features(test)

baseline_features = ["Sex", "Pclass", "FamilySize", "AgeGroup"]

X_baseline = train_fe[baseline_features]
X_embarked = train_fe[baseline_features + ["Embarked"]]
X_fare = train_fe[baseline_features + ["Fare"]]
X_embarked_fare = train_fe[baseline_features + ["Embarked", "Fare"]]

y = train_fe["Survived"]


In [5]:
# 전처리 Pipeline
numerical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)


def build_model_pipeline(num_features, cat_features, estimator):
    preprocessor = ColumnTransformer(
        [
            ("num", numerical_transformer, num_features),
            ("cat", categorical_transformer, cat_features),
        ]
    )

    return Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )


In [6]:
# 모든 실험에서 동일한 CV 분할을 사용한다.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

logistic_model = LogisticRegression(max_iter=1000, random_state=42)

baseline_pipeline = build_model_pipeline(
    ["FamilySize"],
    ["Sex", "Pclass", "AgeGroup"],
    clone(logistic_model),
)
embarked_pipeline = build_model_pipeline(
    ["FamilySize"],
    ["Sex", "Pclass", "AgeGroup", "Embarked"],
    clone(logistic_model),
)
fare_pipeline = build_model_pipeline(
    ["FamilySize", "Fare"],
    ["Sex", "Pclass", "AgeGroup"],
    clone(logistic_model),
)
embarked_fare_pipeline = build_model_pipeline(
    ["FamilySize", "Fare"],
    ["Sex", "Pclass", "AgeGroup", "Embarked"],
    clone(logistic_model),
)


def get_oof_prediction(X, pipeline):
    return cross_val_predict(
        estimator=pipeline,
        X=X,
        y=y,
        cv=cv,
        method="predict",
        n_jobs=-1,
    )


oof_pred_baseline = get_oof_prediction(X_baseline, baseline_pipeline)
oof_pred_embarked = get_oof_prediction(X_embarked, embarked_pipeline)
oof_pred_fare = get_oof_prediction(X_fare, fare_pipeline)
oof_pred_embarked_fare = get_oof_prediction(
    X_embarked_fare,
    embarked_fare_pipeline,
)


In [7]:
def result_df(model_name, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()

    return pd.DataFrame(
        [
            {
                "model": model_name,
                "TN": tn,
                "FP": fp,
                "FN": fn,
                "TP": tp,
                "accuracy": accuracy_score(y, pred),
                "recall": recall_score(y, pred),
                "precision": precision_score(y, pred),
                "f1": f1_score(y, pred),
                "predicted_survivors": pred.sum(),
            }
        ]
    )


result_baseline = result_df("baseline", oof_pred_baseline)
result_embarked = result_df("+Embarked", oof_pred_embarked)
result_fare = result_df("+Fare", oof_pred_fare)
result_embarked_fare = result_df(
    "+Embarked, +Fare",
    oof_pred_embarked_fare,
)

feature_comparison = pd.concat(
    [
        result_baseline,
        result_embarked,
        result_fare,
        result_embarked_fare,
    ],
    ignore_index=True,
)

feature_comparison


,model,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,baseline,482,67,100,242,0.812570,0.707602,0.783172,0.743472,309
1,+Embarked,481,68,98,244,0.813692,0.713450,0.782051,0.746177,312
2,+Fare,489,60,107,235,0.812570,0.687135,0.796610,0.737834,295
3,"+Embarked, +Fare",481,68,100,242,0.811448,0.707602,0.780645,0.742331,310


### Feature 비교 해석

Embarked 추가 시 정확도는 0.812570에서 0.813692로 변했고, 맞힌 승객 수는 1명 늘었다.
Fare 추가 시 FP는 줄지만 FN은 늘었고, 둘을 함께 추가한 경우 정확도는 0.811448이었다.
이 한 차례의 비교만으로 일반적 효과를 판단하기 어렵다고 보고 기존 네 변수를 유지했다.


## 3. Logistic Regression과 Gradient Boosting 비교

이전 실험에서 Decision Tree, Random Forest, Extra Trees는 이미 비교했다. 이번에는 동일한 baseline feature로 Gradient Boosting을 추가 비교한다.

Random Forest가 여러 트리를 비교적 독립적으로 학습해 결과를 종합한다면, Gradient Boosting은 작은 트리를 순차적으로 추가하면서 앞선 모델의 오류를 보완한다. 이를 통해 Logistic Regression이 직접 표현하지 못한 비선형 관계나 feature 간 상호작용을 학습할 수 있다.


In [8]:
gradient_boosting = GradientBoostingClassifier(random_state=42)

gb_pipeline = build_model_pipeline(
    ["FamilySize"],
    ["Sex", "Pclass", "AgeGroup"],
    gradient_boosting,
)

gb_oof_pred = cross_val_predict(
    estimator=gb_pipeline,
    X=X_baseline,
    y=y,
    cv=cv,
    method="predict",
    n_jobs=-1,
)

gb_result = result_df("Gradient Boosting", gb_oof_pred)

model_comparison = pd.concat(
    [result_baseline, gb_result],
    ignore_index=True,
)

model_comparison


,model,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,baseline,482,67,100,242,0.812570,0.707602,0.783172,0.743472,309
1,Gradient Boosting,491,58,101,241,0.821549,0.704678,0.806020,0.751950,299


### 모델 비교 해석

Gradient Boosting의 OOF accuracy는 0.821549로 기본 로지스틱 회귀의 0.812570보다 높았다.
FP는 67명에서 58명으로 줄고 FN은 100명에서 101명으로 늘어, 전체 오분류는 8명 줄었다.
이 결과에서는 accuracy·precision·F1이 높아졌지만, 비선형 관계가 존재한다는 별도 검정을 한 것은 아니다.


## 4. Gradient Boosting Threshold 비교


In [9]:
gb_oof_proba = cross_val_predict(
    estimator=gb_pipeline,
    X=X_baseline,
    y=y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

thresholds = [0.55, 0.50, 0.475, 0.45, 0.40, 0.35, 0.30]
threshold_results = []

for threshold in thresholds:
    threshold_pred = (gb_oof_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, threshold_pred).ravel()

    threshold_results.append(
        {
            "threshold": threshold,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "accuracy": accuracy_score(y, threshold_pred),
            "recall": recall_score(y, threshold_pred),
            "precision": precision_score(
                y,
                threshold_pred,
                zero_division=0,
            ),
            "f1": f1_score(y, threshold_pred),
            "predicted_survivors": threshold_pred.sum(),
        }
    )

threshold_summary = pd.DataFrame(threshold_results)
threshold_summary


,threshold,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,0.550,499,50,111,231,0.819304,0.675439,0.822064,0.741573,281
1,0.500,491,58,101,241,0.821549,0.704678,0.806020,0.751950,299
2,0.475,481,68,96,246,0.815937,0.719298,0.783439,0.750000,314
3,0.450,480,69,95,247,0.815937,0.722222,0.781646,0.750760,316
4,0.400,459,90,85,257,0.803591,0.751462,0.740634,0.746009,347
5,0.350,435,114,68,274,0.795735,0.801170,0.706186,0.750685,388
6,0.300,421,128,63,279,0.785634,0.815789,0.685504,0.744993,407


### Threshold 비교 해석

비교한 Gradient Boosting 기준 중 0.50의 **accuracy와 F1**이 가장 높았다.
Precision은 0.55에서 0.822064로, 0.50의 0.806020보다 높았다.
0.50에서 0.45로 낮추면 recall은 높아지지만 accuracy·precision·F1은 낮아졌다.
이 단계에서는 accuracy를 기준으로 0.50을 제출 후보로 유지했다.


## 5. 전체 train 학습과 Kaggle 제출 파일 생성

`cross_val_predict()`는 각 fold의 OOF 예측을 만들기 위한 함수이므로, Kaggle 제출 전에는 선택한 pipeline을 전체 train 데이터로 다시 학습한다.

비교를 위해 다음 세 파일을 생성한다.

1. Gradient Boosting, threshold `0.5`
2. Logistic Regression, threshold `0.5`
3. Logistic Regression, threshold `0.45`

이 셀은 제출 파일의 모델·threshold·경로가 일치하도록 정리한 코드다. 2026-10-06 재실행에서 세 파일을 새로 만들었다.
생존 예측 수(151·145·166)는 정리 전에 첨부했던 CSV와 같지만, 행 단위로 같은 파일인지는 대조하지 않았다.


In [10]:
final_features = ["Sex", "FamilySize", "Pclass", "AgeGroup"]

X_final = train_fe[final_features]
y_final = train_fe["Survived"]
X_test_final = test_fe[final_features]


def make_submission(pipeline, threshold, file_name):
    final_pipeline = clone(pipeline)
    final_pipeline.fit(X_final, y_final)

    test_proba = final_pipeline.predict_proba(X_test_final)[:, 1]
    test_prediction = (test_proba >= threshold).astype(int)

    submission = pd.DataFrame(
        {
            "PassengerId": test["PassengerId"],
            "Survived": test_prediction,
        }
    )

    if Path("/kaggle/working").is_dir():
        output_dir = Path("/kaggle/working/submissions")
    else:
        repo_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
        output_dir = repo_dir / "submissions"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / file_name
    submission.to_csv(output_path, index=False)

    print(file_name)
    print("shape:", submission.shape)
    print("missing values:\n", submission.isna().sum())
    print("prediction counts:\n", submission["Survived"].value_counts())
    print("saved to:", output_path)
    print()

    return submission, output_path


gb_submission, gb_output_path = make_submission(
    gb_pipeline,
    threshold=0.5,
    file_name="submission_gradient_boosting_050.csv",
)

lr_submission_050, lr_output_path_050 = make_submission(
    baseline_pipeline,
    threshold=0.5,
    file_name="submission_logistic_regression_050.csv",
)

lr_submission_045, lr_output_path_045 = make_submission(
    baseline_pipeline,
    threshold=0.45,
    file_name="submission_logistic_regression_045.csv",
)


submission_gradient_boosting_050.csv
shape: (418, 2)
missing values:
 PassengerId    0
Survived       0
dtype: int64
prediction counts:
 Survived
0    267
1    151
Name: count, dtype: int64
saved to: /content/submissions/submission_gradient_boosting_050.csv

submission_logistic_regression_050.csv
shape: (418, 2)
missing values:
 PassengerId    0
Survived       0
dtype: int64
prediction counts:
 Survived
0    273
1    145
Name: count, dtype: int64
saved to: /content/submissions/submission_logistic_regression_050.csv

submission_logistic_regression_045.csv
shape: (418, 2)
missing values:
 PassengerId    0
Survived       0
dtype: int64
prediction counts:
 Survived
0    252
1    166
Name: count, dtype: int64
saved to: /content/submissions/submission_logistic_regression_045.csv



## 6. 당시 기록한 Kaggle 제출 점수

아래는 원본 노트북에 적힌 당시 기록이다. 이번 정리에서는 Kaggle 제출 이력과 대조하지 않았고,
첨부 CSV를 각 점수에 확정적으로 연결하지 않았다. 재실행 후 다시 제출하지는 않았다.

| 모델 | Threshold | OOF Accuracy | 기록된 Kaggle 점수 |
|---|---:|---:|---:|
| Gradient Boosting | 0.50 | 0.82155 | 0.76794 |
| Logistic Regression | 0.50 | 0.81257 | 0.77272 |
| Logistic Regression | 0.45 | 0.80696 | 0.75598 |

기록상 최종 선택은 Logistic Regression, threshold 0.50이었다.
OOF에서 앞선 후보와 제출 점수가 앞선 후보가 같지 않았다.
작은 차이를 일반적 우열의 증거로 삼지 않으며, 제출 점수를 본 뒤 선택했으므로
그 점수를 독립적인 최종 평가처럼 설명하지 않는다.


## 7. 이 단계에서 남긴 결론

변수·모델·분류 기준을 바꾸면 개선되는 지표와 나빠지는 지표가 달랐다.
어떤 지표를 우선할지 정하지 않고 가장 좋은 모델 하나를 선언하기는 어렵다.
또한 같은 검증 결과와 제출 점수를 반복해 보며 선택한 결과에는 선택 편향이 생길 수 있다.

이후 확인 후보였던 Title 추가 실험은 이 저장소에서 완료했다고 주장하지 않는다.
현재 공개 범위는 남아 있는 실험 기록의 재구성과 해석이다.


## 실행 환경 기록

전체 재실행 시 아래 버전을 결과와 함께 보관합니다.


In [11]:
import platform
from importlib.metadata import version

print("Python", platform.python_version())
for package in ["numpy", "pandas", "scikit-learn", "matplotlib"]:
    print(package, version(package))


Python 3.13.16
numpy 2.1.3
pandas 2.2.3
scikit-learn 1.6.1
matplotlib 3.10.0
